# Running a Finance Company with CrewAI Flows and Parallel Agents

In the podcaster tutorial, you built sequential crews. Now you'll orchestrate an entire company: departments that work in parallel, hand off results through shared state, and route decisions conditionally.

| Step | Concept | What It Unlocks |
|------|---------|-----------------|
| 1 | Setup | Reuse the same CrewAI + OpenRouter pattern |
| 2 | Flow + `@start` + `@listen` | Event-driven orchestration |
| 3 | Pydantic Flow State | Shared memory across departments |
| 4 | Crew inside a Flow step | A department becomes a flow method |
| 5 | `@listen` | Sequential handoffs between departments |
| 6 | Parallel `@start` + `and_()` | Departments working at the same time |
| 7 | `@router` | Risk-based branching |
| 8 | `or_()` | Merge multiple paths into one final step |
| 9 | Complete company flow | A full finance-company workflow |


---

## Step 1: Setup

This notebook assumes you already know the CrewAI basics from the podcaster project: `Agent`, `Task`, `Crew`, tools, and sequential task context.

The new imports are the Flow primitives:

- `Flow`: the class that owns the workflow
- `@start`: marks entry points
- `@listen`: runs after another step emits an event
- `@router`: chooses a branch by returning a label
- `and_`: waits for multiple steps to finish
- `or_`: proceeds when any listed path completes


In [1]:
# !pip install -q "crewai[tools]" python-dotenv pydantic

import os
from datetime import datetime
from time import perf_counter

from crewai import Agent, Crew, Process, Task
from crewai.flow.flow import Flow, and_, listen, or_, router, start
from crewai.llm import LLM
from crewai_tools import EXASearchTool
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv("../../.env")

llm = LLM(
    model="openai/gpt-4.1-2025-04-14",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
search_tool = EXASearchTool()


/Users/ishandutta/miniconda3/envs/agents/lib/python3.10/site-packages/crewai_tools/tools/exa_tools/exa_search_tool.py:40: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  api_key: Optional[str] = Field(


---

## Step 2: What is a Flow?

A **Crew** is a team of agents doing work.

A **Flow** is the manager that decides which work happens first, what happens next, and which branch to take.

Let's start with the smallest possible Flow: one step starts, another step listens for it.


In [ ]:
class TinyCompanyFlow(Flow):
    @start()
    def open_office(self):
        print("Office opened")
        return "market opens"

    @listen(open_office)
    def morning_meeting(self, opening_message):
        print(f"Morning meeting starts because: {opening_message}")
        return "meeting complete"


tiny_flow = TinyCompanyFlow()
await tiny_flow.kickoff_async()


---

## Step 3: Flow State

In a sequential Crew, one task passes context to the next task with `context=[previous_task]`.

In a Flow, every method can read and write the same state object. Here, the state is the company's shared operating memory.


In [5]:
from tabnanny import verbose


class FinanceCompanyState(BaseModel):
    company: str = Field(default="NVIDIA", description="Company being evaluated")
    market_research: str = ""
    risk_assessment: str = ""
    risk_level: str = "normal"
    compliance_notes: str = ""
    portfolio_strategy: str = ""
    client_report: str = ""


class StateDemoFlow(Flow[FinanceCompanyState]):
    @start()
    def inspect_state(self):
        print(f"Company under review: {self.state.company}")
        self.state.market_research = "Market research will be stored here."
        print(self.state.market_research)


state_demo = StateDemoFlow()
await state_demo.kickoff_async()

╭──────────────────────────────────────────────── Flow Execution ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: StateDemoFlow                                                                                            │
│  ID: 2bd55ae2-6621-4948-a9dd-3bcec2e86763                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Flow started with ID: 2bd55ae2-6621-4948-a9dd-3bcec2e86763

Output()

Company under review: NVIDIA

Market research will be stored here.

╭──────────────────────────────────────────────── Flow Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: StateDemoFlow                                                                                            │
│  ID: 2bd55ae2-6621-4948-a9dd-3bcec2e86763                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---

## Step 4: First Department in a Flow

Now the Flow starts doing real work. The first department is **Market Research**.

The pattern is:

1. Define one agent
2. Define one task
3. Wrap them in a one-agent Crew
4. Run the Crew inside a Flow method
5. Save the result into `self.state`


In [6]:
class MarketResearchFlow(Flow[FinanceCompanyState]):
    @start()
    def run_market_research(self):
        researcher = Agent(
            role="Market Researcher",
            goal="Research the latest market position, business momentum, and investor narrative for {company}",
            backstory="You are a finance research analyst who finds concise, source-grounded market context.",
            llm=llm,
            tools=[search_tool],
            verbose=True,
            max_iter=3,
            max_rpm=8,
            max_execution_time=180,
            respect_context_window=True,
        )

        research_task = Task(
            description=(
                "Research {company} as of {current_month} {current_year}. "
                "Find 3 important market developments, summarize each one, and explain why investors care."
            ),
            expected_output=(
                "A concise markdown brief with: 3 market developments, investor relevance, and source names or URLs."
            ),
            agent=researcher,
        )

        research_crew = Crew(
            agents=[researcher],
            tasks=[research_task],
            process=Process.sequential,
            verbose=True,
            cache=True,
        )

        result = research_crew.kickoff(
            inputs={
                "company": self.state.company,
                "current_month": datetime.now().strftime("%B"),
                "current_year": str(datetime.now().year),
            }
        )
        self.state.market_research = str(result)
        return self.state.market_research


market_flow = MarketResearchFlow()
market_flow.state.company = "NVIDIA"
await market_flow.kickoff_async()


╭──────────────────────────────────────────────── Flow Execution ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: MarketResearchFlow                                                                                       │
│  ID: 5a67fcdf-3514-4d3b-b1ae-3a49c2bc8a98                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Flow started with ID: 5a67fcdf-3514-4d3b-b1ae-3a49c2bc8a98

Output()

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b75d4a23-0688-4d3d-a657-c12b8fd1f502                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Researcher                                                                                       │
│                                                                                                                 │
│  Task: Research NVIDIA as of May 2026. Find 3 important market developments, summarize each one, and explain    │
│  why investors care.                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 0b46e5d5-3243-47fa-b3ab-4a3590156a75                                                                     │
│  Agent: Market Researcher                                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b75d4a23-0688-4d3d-a657-c12b8fd1f502                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ---                                                                                              │
│  ## NVIDIA Market Brief — May 2026                                                                              │
│                                                                                                                 │
│  ### 1. $40 Billion AI Investment Surge                                                                         │
│  **Summary:**                                                                                                   │
│  NVIDIA has topped $40 billion in equity bets and reinforced its leading position in the AI ecosystem,          │
│  embracing a wave of investment from institutional and retail investors. The company is leveraging these funds  │
│  to further expand its R&D and large-scale AI infrastructure projects.                                          │
│                                                                                                                 │
│  **Investor Relevance:**                                                                                        │
│  This surge solidifies NVIDIA’s central position in the AI hardware and software supply chain, underlining      │
│  confidence from the investment community and supporting continued high valuation multiples. It reassures       │
│  investors that NVIDIA remains the strategic beneficiary of global AI expansion.                                │
│                                                                                                                 │
│  **Source:**                                                                                                    │
│  [CNBC - Nvidia embraces AI investor, topping $40 billion in equity bets (May                                   │
│  2026)](https://www.cnbc.com/2026/05/09/nvidia-embraces-ai-investor-topping-40-billion-in-equity-bets-2026.htm  │
│  l)                                                                                                             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Long-Term U.S. Manufacturing Partnership with Corning                                                   │
│  **Summary:**                                                                                                   │
│  NVIDIA and Corning have announced a long-term strategic partnership aimed at strengthening U.S. manufacturing  │
│  for AI infrastructure, focusing on supply chain resilience and advanced component production within the        │
│  United States.                                                                                                 │
│                                                                                                                 │
│  **Investor Relevance:**                                                                                        │
│  By securing domestic manufacturing capabilities, NVID

Output()

╭──────────────────────────────────────────────── Flow Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: MarketResearchFlow                                                                                       │
│  ID: 5a67fcdf-3514-4d3b-b1ae-3a49c2bc8a98                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

'---\n## NVIDIA Market Brief — May 2026\n\n### 1. $40 Billion AI Investment Surge\n**Summary:**  \nNVIDIA has topped $40 billion in equity bets and reinforced its leading position in the AI ecosystem, embracing a wave of investment from institutional and retail investors. The company is leveraging these funds to further expand its R&D and large-scale AI infrastructure projects.\n\n**Investor Relevance:**  \nThis surge solidifies NVIDIA’s central position in the AI hardware and software supply chain, underlining confidence from the investment community and supporting continued high valuation multiples. It reassures investors that NVIDIA remains the strategic beneficiary of global AI expansion.\n\n**Source:**  \n[CNBC - Nvidia embraces AI investor, topping $40 billion in equity bets (May 2026)](https://www.cnbc.com/2026/05/09/nvidia-embraces-ai-investor-topping-40-billion-in-equity-bets-2026.html)\n\n---\n\n### 2. Long-Term U.S. Manufacturing Partnership with Corning\n**Summary:**  \nNVI

---

## Step 5: Chain with `@listen`

A listener runs after another method completes.

Here, Risk Assessment waits for Market Research. This is useful when the second department depends on the first department's output.


In [7]:
class SequentialFinanceFlow(Flow[FinanceCompanyState]):
    @start()
    def run_market_research(self):
        researcher = Agent(
            role="Market Researcher",
            goal="Research recent market developments for {company}",
            backstory="You find concise, source-grounded finance updates for investors.",
            llm=llm,
            tools=[search_tool],
            verbose=True,
            max_iter=3,
            max_rpm=8,
            max_execution_time=180,
            respect_context_window=True,
        )

        task = Task(
            description="Find 3 recent market developments about {company} and explain why investors care.",
            expected_output="A concise markdown market brief with sources.",
            agent=researcher,
        )

        crew = Crew(agents=[researcher], tasks=[task], process=Process.sequential, verbose=True, cache=True)
        result = crew.kickoff(inputs={"company": self.state.company})
        self.state.market_research = str(result)
        return self.state.market_research

    @listen(run_market_research)
    def run_risk_assessment(self, market_research):
        risk_analyst = Agent(
            role="Risk Analyst",
            goal="Identify the biggest investment risks for {company}",
            backstory="You are a careful risk analyst who flags valuation, market, regulatory, and execution risks.",
            llm=llm,
            tools=[search_tool],
            verbose=True,
            max_iter=3,
            max_rpm=8,
            max_execution_time=180,
            respect_context_window=True,
        )

        task = Task(
            description=(
                "Review this market research and search for current risks for {company}:\n\n"
                "{market_research}\n\n"
                "End your answer with exactly one line: Risk Level: normal OR Risk Level: high_risk."
            ),
            expected_output="A concise risk brief ending with Risk Level: normal or Risk Level: high_risk.",
            agent=risk_analyst,
        )

        crew = Crew(agents=[risk_analyst], tasks=[task], process=Process.sequential, verbose=True, cache=True)
        result = crew.kickoff(inputs={"company": self.state.company, "market_research": market_research})
        self.state.risk_assessment = str(result)
        self.state.risk_level = "high_risk" if "risk level: high_risk" in self.state.risk_assessment.lower() else "normal"
        return self.state.risk_assessment


sequential_flow = SequentialFinanceFlow()
sequential_flow.state.company = "NVIDIA"
sequential_start = perf_counter()
await sequential_flow.kickoff_async()
sequential_seconds = perf_counter() - sequential_start
print(f"Sequential flow finished in {sequential_seconds:.1f} seconds")


╭──────────────────────────────────────────────── Flow Execution ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: SequentialFinanceFlow                                                                                    │
│  ID: 09ac3ead-aa16-416a-92d3-df98b19cd445                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Flow started with ID: 09ac3ead-aa16-416a-92d3-df98b19cd445

Output()

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9f48c104-1fa7-4fc3-a9c3-943c5e490742                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Researcher                                                                                       │
│                                                                                                                 │
│  Task: Find 3 recent market developments about NVIDIA and explain why investors care.                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: ed7a4170-45de-43bb-84d7-827b5e16e41c                                                                     │
│  Agent: Risk Analyst                                                                                            │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3189c9ee-73d8-42d7-a113-e3e41213e23a                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: # NVIDIA: Key Investment Risks (As of December 2024)                                             │
│                                                                                                                 │
│  ## 1. Valuation Risk                                                                                           │
│  NVIDIA trades at premium valuations, with persistent debate among analysts about whether earnings growth       │
│  justifies its price. The stock’s momentum, driven partly by retail interest and AI hype, leaves little margin  │
│  for error. Even as fair value estimates are raised, any sign of slower earnings growth or                      │
│  weaker-than-expected AI adoption could trigger sharp corrections.[Morningstar][ProactiveInvestors]             │
│                                                                                                                 │
│  ## 2. Market Risk & Customer Concentration                                                                     │
│  While NVIDIA dominates the GPU sector, ~36% of its Q3 2024 revenue came from just three unnamed customers.     │
│  This creates concentration risk: changes in procurement strategies, regulatory issues, or competitive          │
│  responses at these entities might materially impact NVIDIA’s top line.[Motley Fool]                            │
│                                                                                                                 │
│  Additionally, broader industry GPU shipments declined by over 14% despite NVIDIA’s gains—signaling fragility   │
│  in gaming/video hardware markets. Heavy reliance on data center and AI segments makes NVIDIA exposed to        │
│  cyclical shifts or disruptive technologies.                                                                    │
│                                                                                                                 │
│  ## 3. Execution Risk: Supply Chain and Delivery Challenges                                                     │
│  Despite surging demand, NVIDIA has recently faced "supply snags" that threaten to limit GPU deliveries. Any    │
│  aggravation—such as shortages in advanced manufacturing substrates—can restrict order fulfillment, hurt        │
│  customer relationships, or invite competitors to fill gaps.[Reuters]                                           │
│                                                                                                                 │
│  ## 4. Regulatory and Geopolitical Risk                                                                         │
│  The regulatory climate is becoming significantly more hostile:                                                 │
│  - China has launched an antitrust probe into NVIDIA, escalating chip sector tensions with the US. Potential    │
│  penalties, forced structural changes, or loss of access to Chinese markets would be highly material for        │
│  NVIDIA.[Reuters][IG][24/7 Wall St.]                                                                            │
│  - Ongoing US export restrictions on advanced chips to China may constrain growth in one of NVIDIA’s            │
│  historically important end-markets.                  

Output()

╭──────────────────────────────────────────────── Flow Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: SequentialFinanceFlow                                                                                    │
│  ID: 09ac3ead-aa16-416a-92d3-df98b19cd445                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Sequential flow finished in 17.5 seconds


---

## Step 6: Go Parallel with Multiple `@start` Methods and `and_()`

Market Research and Risk Assessment do not have to wait for each other. In a real company, those departments can work at the same time.

CrewAI Flows support that pattern with multiple `@start()` methods.

Then `and_(...)` makes Portfolio Strategy wait until **both** departments finish.


In [8]:
def build_market_research_crew():
    researcher = Agent(
        role="Market Researcher",
        goal="Research recent market developments for {company}",
        backstory="You find concise, source-grounded finance updates for investors.",
        llm=llm,
        tools=[search_tool],
        verbose=True,
        max_iter=3,
        max_rpm=8,
        max_execution_time=180,
        respect_context_window=True,
    )
    task = Task(
        description="Find 3 recent market developments about {company} and explain why investors care.",
        expected_output="A concise markdown market brief with sources.",
        agent=researcher,
    )
    return Crew(agents=[researcher], tasks=[task], process=Process.sequential, verbose=True, cache=True)


def build_risk_assessment_crew():
    risk_analyst = Agent(
        role="Risk Analyst",
        goal="Identify the biggest investment risks for {company}",
        backstory="You are a careful risk analyst who flags valuation, market, regulatory, and execution risks.",
        llm=llm,
        tools=[search_tool],
        verbose=True,
        max_iter=3,
        max_rpm=8,
        max_execution_time=180,
        respect_context_window=True,
    )
    task = Task(
        description=(
            "Find the current investment risks for {company}. "
            "Focus on valuation, competition, regulation, demand, and execution. "
            "End your answer with exactly one line: Risk Level: normal OR Risk Level: high_risk."
        ),
        expected_output="A concise risk brief ending with Risk Level: normal or Risk Level: high_risk.",
        agent=risk_analyst,
    )
    return Crew(agents=[risk_analyst], tasks=[task], process=Process.sequential, verbose=True, cache=True)


def build_portfolio_strategy_crew():
    strategist = Agent(
        role="Investment Strategist",
        goal="Turn market research and risk assessment into a portfolio recommendation for {company}",
        backstory="You convert research and risk signals into clear investment strategy.",
        llm=llm,
        verbose=True,
        max_iter=3,
        max_rpm=8,
        max_execution_time=180,
        respect_context_window=True,
    )
    task = Task(
        description=(
            "Create a portfolio strategy for {company}.\n\n"
            "Market Research:\n{market_research}\n\n"
            "Risk Assessment:\n{risk_assessment}\n\n"
            "Recommend one stance: Buy, Hold, Watchlist, or Avoid. Explain the reasoning."
        ),
        expected_output="A concise strategy memo with stance, reasoning, and key watch items.",
        agent=strategist,
    )
    return Crew(agents=[strategist], tasks=[task], process=Process.sequential, verbose=True, cache=True)


class ParallelFinanceFlow(Flow[FinanceCompanyState]):
    @start()
    def run_market_research(self):
        result = build_market_research_crew().kickoff(inputs={"company": self.state.company})
        self.state.market_research = str(result)
        return self.state.market_research

    @start()
    def run_risk_assessment(self):
        result = build_risk_assessment_crew().kickoff(inputs={"company": self.state.company})
        self.state.risk_assessment = str(result)
        self.state.risk_level = "high_risk" if "risk level: high_risk" in self.state.risk_assessment.lower() else "normal"
        return self.state.risk_assessment

    @listen(and_(run_market_research, run_risk_assessment))
    def run_portfolio_strategy(self):
        result = build_portfolio_strategy_crew().kickoff(
            inputs={
                "company": self.state.company,
                "market_research": self.state.market_research,
                "risk_assessment": self.state.risk_assessment,
            }
        )
        self.state.portfolio_strategy = str(result)
        return self.state.portfolio_strategy


parallel_flow = ParallelFinanceFlow()
parallel_flow.state.company = "NVIDIA"
parallel_start = perf_counter()
await parallel_flow.kickoff_async()
parallel_seconds = perf_counter() - parallel_start
print(f"Parallel flow finished in {parallel_seconds:.1f} seconds")


╭──────────────────────────────────────────────── Flow Execution ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ParallelFinanceFlow                                                                                      │
│  ID: 5f7121f6-2a16-4978-91be-7d1f506f81d6                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Flow started with ID: 5f7121f6-2a16-4978-91be-7d1f506f81d6

Output()

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7c865805-0f09-4395-ae57-416b34500bed                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Market Researcher                                                                                       │
│                                                                                                                 │
│  Task: Find 3 recent market developments about NVIDIA and explain why investors care.                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 9d9a703c-4c09-4696-9e1c-5b88e72f64b4                                                                     │
│  Agent: Investment Strategist                                                                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9e47928e-8308-414b-ab23-fb3b25836732                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ---                                                                                              │
│                                                                                                                 │
│  **NVIDIA (NVDA) Portfolio Strategy Memo – June 2024**                                                          │
│                                                                                                                 │
│  **Stance:** Hold                                                                                               │
│                                                                                                                 │
│  **Rationale:**                                                                                                 │
│  NVIDIA’s fundamental leadership in AI hardware and its transformation of the data center economy remains       │
│  unchallenged. However, after a meteoric run that briefly made it the world’s most valuable company, NVDA has   │
│  entered a technical correction, dropping over 13% in just three days and erasing $550 billion in market cap.   │
│  Valuations are stretched, with price-to-earnings multiples at historic, demanding levels. Continued            │
│  outperformance is increasingly hard to justify without another period of hypergrowth in AI spending, leaving   │
│  the stock highly vulnerable to any negative surprises.                                                         │
│                                                                                                                 │
│  While AI infrastructure demand from cloud giants and enterprises is robust, several factors now elevate risk:  │
│                                                                                                                 │
│  - **Valuation Risk:** Current multiples price in perfection. Even a modest deceleration in revenue/EPS growth  │
│  could force a sharp market re-rating.                                                                          │
│  - **Competitive Threats:** AMD, Intel, and cloud custom silicon are accelerating their GPU and AI chip         │
│  ambitions. While NVIDIA maintains the lead, competitive pressures are rising and may eventually compress       │
│  margins or erode share.                                                                                        │
│  - **Regulatory Uncertainty:** Export restrictions, especially in the U.S.-China technology rivalry, could      │
│  constrain access to important markets or product lines, adversely impacting growth.                            │
│  - **Demand Normalization:** If AI investment slows or customer concentration at hyperscalers turns adverse,    │
│  top-line growth will be at risk.                                                                               │
│  - **Execution Risk:** NVIDIA’s scale and complexity increases the risk from supply chain, R&D execution, or    │
│  new product ramp issues.                                                                                       │
│                                                                                                                 │
│  Despite these headwinds, NVIDIA’s exceptionally stron

Output()

╭──────────────────────────────────────────────── Flow Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ParallelFinanceFlow                                                                                      │
│  ID: 5f7121f6-2a16-4978-91be-7d1f506f81d6                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Parallel flow finished in 29.5 seconds


---

## Step 7: Conditional Routing with `@router`

A router turns a workflow from a straight line into a decision tree.

Here, the Portfolio Strategy step finishes, then the router checks `self.state.risk_level`:

- `high_risk` sends the report through Compliance Review
- `approved` skips Compliance and moves toward the client report


In [9]:
class RoutedFinanceFlow(ParallelFinanceFlow):
    @router(ParallelFinanceFlow.run_portfolio_strategy)
    def route_on_risk(self):
        if self.state.risk_level == "high_risk":
            return "high_risk"
        return "approved"

    @listen("high_risk")
    def run_compliance_review(self):
        self.state.compliance_notes = (
            "Compliance reviewed the strategy because risk was marked high. "
            "Client-facing language should avoid guarantees, highlight uncertainty, and frame the stance as research commentary."
        )
        print("Compliance review completed")
        return self.state.compliance_notes

    @listen("approved")
    def approved_path(self):
        self.state.compliance_notes = "No extra compliance review required."
        print("Strategy approved without compliance review")
        return self.state.compliance_notes


routed_flow = RoutedFinanceFlow()
routed_flow.state.company = "NVIDIA"
await routed_flow.kickoff_async()
print(f"Risk route: {routed_flow.state.risk_level}")
print(routed_flow.state.compliance_notes)


╭──────────────────────────────────────────────── Flow Execution ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: RoutedFinanceFlow                                                                                        │
│  ID: 099e55ad-c014-4670-b8aa-2a23dc2785f2                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Flow started with ID: 099e55ad-c014-4670-b8aa-2a23dc2785f2

Output()

╭──────────────────────────────────────────────── Flow Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: RoutedFinanceFlow                                                                                        │
│  ID: 099e55ad-c014-4670-b8aa-2a23dc2785f2                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Risk route: normal



---

## Step 8: Merge Paths with `or_()`

The Client Advisory department should run after either branch is complete:

- the high-risk path after Compliance Review
- the approved path when no Compliance Review is needed

That is what `or_()` gives us: continue after **any one** of the listed events happens.


In [10]:
def build_client_advisory_crew():
    advisor = Agent(
        role="Client Advisor",
        goal="Write a clear investor-facing report for {company}",
        backstory="You translate internal research into plain-English client recommendations.",
        llm=llm,
        verbose=True,
        max_iter=3,
        max_rpm=8,
        max_execution_time=180,
        respect_context_window=True,
    )
    task = Task(
        description=(
            "Write a client advisory report for {company}.\n\n"
            "Market Research:\n{market_research}\n\n"
            "Risk Assessment:\n{risk_assessment}\n\n"
            "Portfolio Strategy:\n{portfolio_strategy}\n\n"
            "Compliance Notes:\n{compliance_notes}\n\n"
            "Keep it concise, balanced, and useful for an investor."
        ),
        expected_output="A client-ready investor report with summary, recommendation, risks, and next steps.",
        agent=advisor,
    )
    return Crew(agents=[advisor], tasks=[task], process=Process.sequential, verbose=True, cache=True)


class AdvisoryFinanceFlow(RoutedFinanceFlow):
    @listen(or_(RoutedFinanceFlow.run_compliance_review, RoutedFinanceFlow.approved_path))
    def run_client_advisory(self):
        result = build_client_advisory_crew().kickoff(
            inputs={
                "company": self.state.company,
                "market_research": self.state.market_research,
                "risk_assessment": self.state.risk_assessment,
                "portfolio_strategy": self.state.portfolio_strategy,
                "compliance_notes": self.state.compliance_notes,
            }
        )
        self.state.client_report = str(result)
        return self.state.client_report


advisory_flow = AdvisoryFinanceFlow()
advisory_flow.state.company = "NVIDIA"
await advisory_flow.kickoff_async()
print(advisory_flow.state.client_report)


╭──────────────────────────────────────────────── Flow Execution ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: AdvisoryFinanceFlow                                                                                      │
│  ID: 0aaf595d-6c93-497e-98fc-65e7f039219c                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Flow started with ID: 0aaf595d-6c93-497e-98fc-65e7f039219c

Output()

╭──────────────────────────────────────────────── Flow Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: AdvisoryFinanceFlow                                                                                      │
│  ID: 0aaf595d-6c93-497e-98fc-65e7f039219c                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---

## Step 9: The Complete Finance Company Flow

This is the same company as one clean Flow class.

```mermaid
flowchart TD
    subgraph parallel [Parallel Start]
        MR["@start: Market Research"]
        RA["@start: Risk Assessment"]
    end

    MR --> PS
    RA --> PS
    PS["@listen and_: Portfolio Strategy"]
    PS --> RT["@router: risk_level?"]

    RT -->|"high_risk"| CR["@listen: Compliance Review"]
    RT -->|"approved"| AP["@listen: Approved Path"]

    CR --> CA["@listen or_: Client Advisory"]
    AP --> CA
    CA --> Report["Investor Report"]
```


In [ ]:
class FinanceCompanyFlow(Flow[FinanceCompanyState]):
    @start()
    def run_market_research(self):
        result = build_market_research_crew().kickoff(inputs={"company": self.state.company})
        self.state.market_research = str(result)
        return self.state.market_research

    @start()
    def run_risk_assessment(self):
        result = build_risk_assessment_crew().kickoff(inputs={"company": self.state.company})
        self.state.risk_assessment = str(result)
        self.state.risk_level = "high_risk" if "risk level: high_risk" in self.state.risk_assessment.lower() else "normal"
        return self.state.risk_assessment

    @listen(and_(run_market_research, run_risk_assessment))
    def run_portfolio_strategy(self):
        result = build_portfolio_strategy_crew().kickoff(
            inputs={
                "company": self.state.company,
                "market_research": self.state.market_research,
                "risk_assessment": self.state.risk_assessment,
            }
        )
        self.state.portfolio_strategy = str(result)
        return self.state.portfolio_strategy

    @router(run_portfolio_strategy)
    def route_on_risk(self):
        if self.state.risk_level == "high_risk":
            return "high_risk"
        return "approved"

    @listen("high_risk")
    def run_compliance_review(self):
        self.state.compliance_notes = (
            "Compliance reviewed the strategy because risk was marked high. "
            "The final report must avoid guarantees, explain uncertainty, and present the recommendation as research commentary."
        )
        return self.state.compliance_notes

    @listen("approved")
    def approved_path(self):
        self.state.compliance_notes = "No extra compliance review required."
        return self.state.compliance_notes

    @listen(or_(run_compliance_review, approved_path))
    def run_client_advisory(self):
        result = build_client_advisory_crew().kickoff(
            inputs={
                "company": self.state.company,
                "market_research": self.state.market_research,
                "risk_assessment": self.state.risk_assessment,
                "portfolio_strategy": self.state.portfolio_strategy,
                "compliance_notes": self.state.compliance_notes,
            }
        )
        self.state.client_report = str(result)
        return self.state.client_report


company_flow = FinanceCompanyFlow()
company_flow.state.company = "NVIDIA"

company_flow.plot()

company_start = perf_counter()
await company_flow.kickoff_async()
company_seconds = perf_counter() - company_start

print(f"Finance company flow finished in {company_seconds:.1f} seconds")
print(f"Risk level: {company_flow.state.risk_level}")
print(company_flow.state.client_report)


---

## Step 10: Recap

| Step | You Learned | CrewAI API |
|------|-------------|------------|
| 1 | Reuse the existing CrewAI setup | `Agent`, `Task`, `Crew`, `LLM`, `EXASearchTool` |
| 2 | A Flow is an event-driven manager | `Flow`, `@start`, `@listen` |
| 3 | Flow state is shared company memory | `Flow[FinanceCompanyState]`, `self.state` |
| 4 | A department can be a Crew inside a Flow method | Crew kickoff inside `@start()` |
| 5 | One department can wait for another | `@listen(previous_step)` |
| 6 | Independent departments can run in parallel | Multiple `@start()` methods, `and_()` |
| 7 | The workflow can branch based on state | `@router`, string route labels |
| 8 | Separate branches can merge into one final step | `or_()` |
| 9 | A full company is just departments plus orchestration | Parallel starts, routed decisions, merged output |

### What You Built

You built a finance-company workflow with four departments:

1. Market Research gathers current company context.
2. Risk Assessment independently evaluates investment risk.
3. Portfolio Strategy waits for both and creates a recommendation.
4. Client Advisory produces the investor-facing report after the appropriate risk path completes.

This is the key shift from the podcaster project: instead of one sequential crew, you now have a Flow that coordinates multiple crews like departments in a company.
